In [1]:
# ============================================================
# PS S6E5 - exp_20260503_007_realmlp_shared001_min
# shared_001 RealMLP / PyTabKit minimum reproduction
#
# Purpose:
# - Reproduce shared_001 RealMLP as-is/minimal
# - Save OOF / pred / submission for correlation and blend
# - No ORIG_PRIOR_* graft yet
# - Normalized_TyreLife is dropped
# ============================================================

In [2]:
# If pytabkit is not installed in the notebook environment, uncomment:
!pip install -q pytabkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 17.7 MB/s eta 0:00:00


In [3]:
import os
import gc
import json
import random
import warnings
from pathlib import Path
from importlib.metadata import version

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder

import torch
import pytabkit
from pytabkit import RealMLP_TD_Classifier

warnings.filterwarnings("ignore")

In [4]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260503_007_realmlp_shared001_min"
    COMPETITION = "PS S6E5 Predicting F1 Pit Stops"
    METRIC = "AUC"

    COMP_BASE = Path("/kaggle/input/competitions/playground-series-s6e5")
    TRAIN_PATH = COMP_BASE / "train.csv"
    TEST_PATH = COMP_BASE / "test.csv"
    SUB_PATH = COMP_BASE / "sample_submission.csv"

    ORIG_PATH = Path(
        "/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv"
    )

    OUTDIR = Path(f"/kaggle/working/{EXP_ID}")

    ID_COL = "id"
    TARGET = "PitNextLap"
    DANGER_COL = "Normalized_TyreLife"

    SEED = 42
    N_SPLITS = 5
    USE_TE = True
    USE_ORIGINAL_ROWS = True

    # shared_001 params
    PARAMS = {
        "random_state": 42,
        "verbosity": 2,
        "val_metric_name": "1-auc_ovr",

        "n_ens": 8,
        "n_epochs": 4,
        "batch_size": 256,
        "use_early_stopping": False,
        "early_stopping_additive_patience": 10,
        "early_stopping_multiplicative_patience": 1,

        "lr": 0.03,
        "wd": 0.018,
        "sq_mom": 0.98,
        "lr_sched": "lin_cos_log_15",
        "first_layer_lr_factor": 0.25,

        "embedding_size": 6,
        "max_one_hot_cat_size": 18,
        "hidden_sizes": [512, 256, 128],
        "act": "silu",
        "p_drop": 0.05,
        "p_drop_sched": "expm4t",

        "plr_hidden_1": 16,
        "plr_hidden_2": 8,
        "plr_act_name": "gelu",
        "plr_lr_factor": 0.1151,
        "plr_sigma": 2.33,

        "ls_eps": 0.01,
        "ls_eps_sched": "sqrt_cos",

        "add_front_scale": False,
        "bias_init_mode": "neg-uniform-dynamic-2",
        "tfms": [
            "one_hot",
            "median_center",
            "robust_scale",
            "smooth_clip",
            "embedding",
            "l2_normalize",
        ],
    }


CFG.OUTDIR.mkdir(parents=True, exist_ok=True)

In [5]:
# ============================================================
# Utilities
# ============================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(CFG.SEED)

print("PyTorch  version:", torch.__version__)
print("PyTabKit version:", version("pytabkit"))
print("CUDA available  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device     :", torch.cuda.get_device_name(0))

PyTorch  version: 2.10.0+cu128
PyTabKit version: 1.7.3
CUDA available  : True
CUDA device     : Tesla T4


In [6]:
# ============================================================
# Load data
# ============================================================

train = pd.read_csv(CFG.TRAIN_PATH)
test = pd.read_csv(CFG.TEST_PATH)
sub = pd.read_csv(CFG.SUB_PATH)
orig = pd.read_csv(CFG.ORIG_PATH)

print("train shape:", train.shape)
print("test shape :", test.shape)
print("sub shape  :", sub.shape)
print("orig shape :", orig.shape)

assert CFG.ID_COL in train.columns
assert CFG.ID_COL in test.columns
assert CFG.TARGET in train.columns
assert CFG.TARGET not in test.columns
assert CFG.TARGET in orig.columns
assert CFG.DANGER_COL in orig.columns

print("\ncompetition target mean:", train[CFG.TARGET].mean())
print("original target mean   :", orig[CFG.TARGET].mean())

train shape: (439140, 16)
test shape : (188165, 15)
sub shape  : (188165, 2)
orig shape : (101371, 16)

competition target mean: 0.19898210137996994
original target mean   : 0.25479673673930414


In [7]:
# ============================================================
# Preprocess features
# ============================================================

# shared_001 policy:
# - drop Normalized_TyreLife
# - keep original target separately
# - original rows are used only in each fold train side
orig = orig.drop(columns=[CFG.DANGER_COL])

y_orig = orig[CFG.TARGET].copy()
orig = orig.drop(columns=[CFG.TARGET])

X = train.drop(columns=[CFG.ID_COL, CFG.TARGET]).copy()
y = train[CFG.TARGET].astype(int).copy()
train_id = train[CFG.ID_COL].copy()

X_test = test.drop(columns=[CFG.ID_COL]).copy()
test_id = test[CFG.ID_COL].copy()

print("\nX init shape     :", X.shape)
print("X_test init shape:", X_test.shape)
print("orig init shape  :", orig.shape)

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("\ninit cat_cols:", len(cat_cols), cat_cols)
print("init num_cols:", len(num_cols), num_cols)


X init shape     : (439140, 14)
X_test init shape: (188165, 14)
orig init shape  : (101371, 14)

init cat_cols: 3 ['Driver', 'Compound', 'Race']
init num_cols: 11 ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change']


In [8]:
# ============================================================
# shared_001 Feature Engineering
# ============================================================

category_map = {}

important_combos = [
    ("Race", "Compound"),
    ("Race", "Year"),
]

def feature_engineering(df: pd.DataFrame, fit: bool = False):
    df = df.copy()

    # Categorize numericals by floor value
    for col in num_cols:
        cat_name = f"{col}_cat_"
        if fit:
            codes, uniques = np.floor(df[col]).factorize()
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = np.floor(df[col]).map(code_map).fillna(-1).astype("int32")

        df[cat_name] = codes.astype(str)

    # Arithmetic interactions
    df["_LapNumber_/_RaceProgress"] = (
        df["LapNumber"] / (df["RaceProgress"] + 1e-6)
    ).astype("float32")

    df["_TyreLife_/_LapNumber"] = (
        df["TyreLife"] / df["LapNumber"].clip(lower=1)
    ).astype("float32")

    # RaceProgress quantile bin
    bin_config = {
        "RaceProgress": [200],
    }

    for col, bins_list in bin_config.items():
        for n_bins in bins_list:
            for strategy in ["quantile"]:
                bin_name = f"{col}_{n_bins}_{strategy}_bin_"

                if fit:
                    kb = KBinsDiscretizer(
                        n_bins=n_bins,
                        encode="ordinal",
                        strategy=strategy,
                        subsample=None,
                    )
                    binned = kb.fit_transform(df[[col]]).ravel().astype("int32")
                    category_map[bin_name] = kb
                else:
                    kb = category_map[bin_name]
                    binned = kb.transform(df[[col]]).ravel().astype("int32")

                df[bin_name] = binned.astype(str)

    # Interaction categories
    combo_names = []

    for col1, col2 in important_combos:
        combo_name = f"{col1}_{col2}_"
        combo_names.append(combo_name)

        combo_series = df[col1].astype(str) + "_" + df[col2].astype(str)

        if fit:
            codes, uniques = combo_series.factorize()
            category_map[combo_name] = uniques
        else:
            uniques = category_map[combo_name]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = combo_series.map(code_map).fillna(-1).astype("int32")

        df[combo_name] = codes.astype(str)

    new_cat_cols = [col for col in df.columns if col.endswith("_")]
    new_num_cols = [col for col in df.columns if col.startswith("_")]

    return df, new_cat_cols, new_num_cols, combo_names


X, new_cat_cols, new_num_cols, combo_names = feature_engineering(X, fit=True)
X_test, _, _, _ = feature_engineering(X_test, fit=False)
orig, _, _, _ = feature_engineering(orig, fit=False)

cat_cols = cat_cols + new_cat_cols
num_cols = num_cols + new_num_cols

print("\nlen(new_cat_cols):", len(new_cat_cols))
print("len(new_num_cols):", len(new_num_cols))
print("combo_names:", combo_names)

print("\nX prep shape     :", X.shape)
print("X_test prep shape:", X_test.shape)
print("orig prep shape  :", orig.shape)

print("\nprep cat_cols:", len(cat_cols))
print("prep num_cols:", len(num_cols))


len(new_cat_cols): 14
len(new_num_cols): 2
combo_names: ['Race_Compound_', 'Race_Year_']

X prep shape     : (439140, 30)
X_test prep shape: (188165, 30)
orig prep shape  : (101371, 30)

prep cat_cols: 17
prep num_cols: 13


In [9]:
# ============================================================
# CV Training
# ============================================================

skf_comp = StratifiedKFold(
    n_splits=CFG.N_SPLITS,
    shuffle=True,
    random_state=CFG.SEED,
)

skf_orig = StratifiedKFold(
    n_splits=CFG.N_SPLITS,
    shuffle=True,
    random_state=CFG.SEED,
)

oof = np.zeros(len(X), dtype=np.float32)
pred = np.zeros(len(X_test), dtype=np.float32)

fold_scores = []
feature_cols_by_fold = []

comp_splits = list(skf_comp.split(X, y))
orig_splits = list(skf_orig.split(orig, y_orig))

for fold, ((tr_idx, va_idx), (or_tr_idx, or_va_idx)) in enumerate(
    zip(comp_splits, orig_splits),
    1,
):
    print("\n" + "=" * 80)
    print(f"Fold {fold}/{CFG.N_SPLITS}")
    print("=" * 80)

    X_tr = X.iloc[tr_idx].copy()
    y_tr = y.iloc[tr_idx].copy().reset_index(drop=True)

    if CFG.USE_ORIGINAL_ROWS:
        orig_tr = orig.iloc[or_tr_idx].copy()
        y_orig_tr = y_orig.iloc[or_tr_idx].copy().reset_index(drop=True)

        X_tr = pd.concat([X_tr.reset_index(drop=True), orig_tr.reset_index(drop=True)], axis=0)
        y_tr = pd.concat([y_tr, y_orig_tr], axis=0).reset_index(drop=True)

    X_va = X.iloc[va_idx].copy()
    y_va = y.iloc[va_idx].copy()

    X_te = X_test.copy()

    # Target encoding on combo columns.
    # This follows shared_001: TE is fit on X_tr, including fold train + original train rows.
    if CFG.USE_TE:
        te_cols = combo_names
        te = TargetEncoder(
            cv=CFG.N_SPLITS,
            smooth="auto",
            shuffle=True,
            random_state=CFG.SEED,
        )

        tr_enc = te.fit_transform(X_tr[te_cols], y_tr)
        va_enc = te.transform(X_va[te_cols])
        te_enc = te.transform(X_te[te_cols])

        te_names = [f"_{col}TE" for col in te_cols]

        X_tr[te_names] = tr_enc
        X_va[te_names] = va_enc
        X_te[te_names] = te_enc

    if fold == 1:
        print("len(FEATURES):", len(X_tr.columns.tolist()))
        print("features:", X_tr.columns.tolist())

    print("train fold shape:", X_tr.shape)
    print("valid fold shape:", X_va.shape)
    print("test  fold shape:", X_te.shape)
    print("train target mean:", float(np.mean(y_tr)))
    print("valid target mean:", float(np.mean(y_va)))

    model = RealMLP_TD_Classifier(**CFG.PARAMS)
    model.fit(X_tr, y_tr, X_va, y_va)

    va_pred = model.predict_proba(X_va)[:, 1].astype(np.float32)
    te_pred = model.predict_proba(X_te)[:, 1].astype(np.float32)

    oof[va_idx] = va_pred
    pred += te_pred / CFG.N_SPLITS

    fold_auc = roc_auc_score(y_va, va_pred)
    fold_scores.append(float(fold_auc))

    print(f"Fold {fold} AUC: {fold_auc:.8f}")

    feature_cols_by_fold.append(X_tr.columns.tolist())

    del model, X_tr, X_va, X_te
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


cv_auc = roc_auc_score(y, oof)

print("\n" + "=" * 80)
print("CV RESULT")
print("=" * 80)
print("CV AUC:", cv_auc)
print("fold_scores:", fold_scores)
print("fold mean:", float(np.mean(fold_scores)))
print("fold std :", float(np.std(fold_scores)))


Fold 1/5
len(FEATURES): 32
features: ['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'Year_cat_', 'PitStop_cat_', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime (s)_cat_', 'LapTime_Delta_cat_', 'Cumulative_Degradation_cat_', 'RaceProgress_cat_', 'Position_Change_cat_', '_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber', 'RaceProgress_200_quantile_bin_', 'Race_Compound_', 'Race_Year_', '_Race_Compound_TE', '_Race_Year_TE']
train fold shape: (432408, 32)
valid fold shape: (87828, 32)
test  fold shape: (188165, 32)
train target mean: 0.2094480213132042
valid target mean: 0.19899121009245344
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', '_LapNumber_/_RaceProgress', '_TyreLife_

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/4: val 1-auc_ovr = 0.054555
Epoch 2/4: val 1-auc_ovr = 0.048187
Epoch 3/4: val 1-auc_ovr = 0.047474
Epoch 4/4: val 1-auc_ovr = 0.045744


`Trainer.fit` stopped: `max_epochs=4` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Fold 1 AUC: 0.95425600

Fold 2/5
train fold shape: (432409, 32)
valid fold shape: (87828, 32)
test  fold shape: (188165, 32)
train target mean: 0.20945216218903862
valid target mean: 0.19897982420184906
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', '_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber', '_Race_Compound_TE', '_Race_Year_TE']
Columns classified as categorical: ['Driver', 'Compound', 'Race', 'Year_cat_', 'PitStop_cat_', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime (s)_cat_', 'LapTime_Delta_cat_', 'Cumulative_Degradation_cat_', 'RaceProgress_cat_', 'Position_Change_cat_', 'RaceProgress_200_quantile_bin_', 'Race_Compound_', 'Race_Year_']


...

Fold 4 AUC: 0.95240422

Fold 5/5
train fold shape: (432409, 32)
valid fold shape: (87828, 32)
test  fold shape: (188165, 32)
train target mean: 0.20944984956372323
valid target mean: 0.19897982420184906
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', '_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber', '_Race_Compound_TE', '_Race_Year_TE']
Columns classified as categorical: ['Driver', 'Compound', 'Race', 'Year_cat_', 'PitStop_cat_', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime (s)_cat_', 'LapTime_Delta_cat_', 'Cumulative_Degradation_cat_', 'RaceProgress_cat_', 'Position_Change_cat_', 'RaceProgress_200_quantile_bin_', 'Race_Compound_', 'Race_Year_']


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/4: val 1-auc_ovr = 0.055399
Epoch 2/4: val 1-auc_ovr = 0.048264
Epoch 3/4: val 1-auc_ovr = 0.047716
Epoch 4/4: val 1-auc_ovr = 0.045905


`Trainer.fit` stopped: `max_epochs=4` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Fold 5 AUC: 0.95409474

CV RESULT
CV AUC: 0.9532701289015783
fold_scores: [0.9542559986894835, 0.9522620682808609, 0.9533620376114241, 0.952404215124125, 0.9540947402091658]
fold mean: 0.9532758119830118
fold std : 0.0008277924445418371


In [10]:
# ============================================================
# Save artifacts
# ============================================================

np.save(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy", oof)
np.save(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy", pred)

# csv versions for quick inspection
pd.DataFrame({
    CFG.ID_COL: train_id,
    CFG.TARGET: oof,
}).to_csv(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.csv", index=False)

sub_out = sub.copy()
target_col = [c for c in sub_out.columns if c != CFG.ID_COL][0]
sub_out[target_col] = pred

sub_path = CFG.OUTDIR / f"submission_{CFG.EXP_ID}.csv"
sub_out.to_csv(sub_path, index=False)

feature_cols = feature_cols_by_fold[0]

feature_df = pd.DataFrame({
    "feature": feature_cols,
    "is_original": False,
    "is_cat_initial": [c in cat_cols for c in feature_cols],
    "is_num_initial": [c in num_cols for c in feature_cols],
    "is_te": [c.endswith("TE") for c in feature_cols],
    "is_combo": [c in combo_names for c in feature_cols],
})
feature_df.to_csv(CFG.OUTDIR / "feature_columns.csv", index=False)

result = {
    "experiment": {
        "id": CFG.EXP_ID,
        "competition": CFG.COMPETITION,
        "metric": CFG.METRIC,
    },
    "data": {
        "train_shape": list(train.shape),
        "test_shape": list(test.shape),
        "original_shape_raw": list(pd.read_csv(CFG.ORIG_PATH, nrows=5).shape),
        "original_path": str(CFG.ORIG_PATH),
        "id_col": CFG.ID_COL,
        "target": CFG.TARGET,
        "danger_col": CFG.DANGER_COL,
        "danger_col_used": False,
        "use_original_rows": CFG.USE_ORIGINAL_ROWS,
        "competition_target_mean": float(y.mean()),
        "original_target_mean": float(y_orig.mean()),
    },
    "features": {
        "initial_cat_cols": cat_cols[:3],
        "initial_num_cols": [c for c in train.drop(columns=[CFG.ID_COL, CFG.TARGET]).columns if c not in cat_cols[:3]],
        "new_cat_cols": new_cat_cols,
        "new_num_cols": new_num_cols,
        "combo_names": combo_names,
        "target_encoding": {
            "enabled": CFG.USE_TE,
            "te_cols": combo_names,
            "te_names": [f"_{col}TE" for col in combo_names],
            "note": "TargetEncoder is fit inside each outer fold on competition fold-train plus original fold-train rows.",
        },
        "total_feature_count": len(feature_cols),
        "feature_columns": feature_cols,
        "notes": [
            "Minimal reproduction of shared_001 RealMLP / PyTabKit.",
            "Normalized_TyreLife is dropped from original before training.",
            "Original rows are appended only to each fold training split.",
            "No ORIG_PRIOR_* features are added in this experiment.",
            "Race_Compound and Race_Year interaction categories are target-encoded.",
            "OOF and pred are saved as npy for later correlation and blend.",
        ],
    },
    "model": {
        "family": "RealMLP",
        "library": "pytabkit",
        "estimator": "RealMLP_TD_Classifier",
        "params": CFG.PARAMS,
        "cv": {
            "scheme": "StratifiedKFold",
            "n_splits": CFG.N_SPLITS,
            "shuffle": True,
            "random_state": CFG.SEED,
        },
    },
    "result": {
        "cv_auc": float(cv_auc),
        "fold_scores": fold_scores,
        "fold_mean": float(np.mean(fold_scores)),
        "fold_std": float(np.std(fold_scores)),
    },
    "artifacts": {
        "outdir": str(CFG.OUTDIR),
        "oof_npy": str(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy"),
        "pred_npy": str(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy"),
        "oof_csv": str(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.csv"),
        "submission": str(sub_path),
        "feature_columns": str(CFG.OUTDIR / "feature_columns.csv"),
        "cv_result": str(CFG.OUTDIR / "cv_result.json"),
    },
}

with open(CFG.OUTDIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("\nSaved artifacts to:", CFG.OUTDIR)
print("Submission:", sub_path)

display(feature_df.head(80))
display(sub_out.head())


Saved artifacts to: /kaggle/working/exp_20260503_007_realmlp_shared001_min
Submission: /kaggle/working/exp_20260503_007_realmlp_shared001_min/submission_exp_20260503_007_realmlp_shared001_min.csv


,feature,is_original,is_cat_initial,is_num_initial,is_te,is_combo
0,Driver,False,True,False,False,False
1,Compound,False,True,False,False,False
2,Race,False,True,False,False,False
3,Year,False,False,True,False,False
4,PitStop,False,False,True,False,False
5,LapNumber,False,False,True,False,False
6,Stint,False,False,True,False,False
7,TyreLife,False,False,True,False,False
8,Position,False,False,True,False,False
9,LapTime (s),False,False,True,False,False


,id,PitNextLap
0,439140,0.004347
1,439141,0.014467
2,439142,0.006019
3,439143,0.266038
4,439144,0.778305
